# ESE 6800 · Tien Dee’s layer selection experiment

**Start here.** Section 14 contains our new experiments. On a fresh Colab GPU, run §§0–3 and the first code cell in §4, then jump to §14. The second initialization cell in §4 and §§5–13 retain Amit’s saved L4 outputs. Section 14 reports measurements from its own GPU session.

# ESE 6800 · Iteration 5: per-input depth from an early prefix probe, and Wanda weight pruning

**Idea.** A JEV request runs the shared prefix before any option is scored. After the first `L_PROBE` prefix layers, a small
linear probe reads the hidden state and predicts whether a **shallow path** (extra layers skipped in the rest of the prefix and in
the branches) will give the **same decision** as the deep path. Safe inputs take the shallow path; the rest run at full depth.

Why this fits JEV models specifically:
* **Free labels.** "Safe" = the shallow path picks the same option as the deep path, measured on unlabeled inputs. No gold answers.
* **The decision is made inside the prefix**, before most of the compute, so nothing is paid twice (unlike a cascade).
* **The answer set is closed**, so "same decision" is well defined (no token-by-token early exit as in LLMs).

**Pairs (deep → shallow) tested**

| Pair | Deep path | Shallow path |
|---|---|---|
| `dense \| p8/b8` | dense | skip 8 layers in prefix and branches (Iteration 3's p8/b8 mask, 1.33×) |
| `dense \| p6/b6` | dense | skip 6 layers in both (Iteration 3's p6/b6 mask, 1.22×) |
| `FastJEV 6 \| +prefix 6` | FastJEV S+P 6 | FastJEV S+P 6, with its 6 masked layers also skipped in the prefix |

**Design**
* Probe training data: 600 AI2D + 600 POPE questions **disjoint from the sweep set** (POPE: no image shared with the sweep set).
* The routing threshold is chosen **on training data only** (out-of-fold), as the most-shallow threshold whose flip rate vs the
  deep path stays under `FLIP_BUDGET`. The 500 + 500 sweep set is then evaluated once at that threshold.
* Controls: static deep, static shallow, **random routing at the same shallow fraction** (does the probe beat chance?) and
  **oracle routing** (upper bound).
* The adaptive path is also run for real, to check the offline simulation and to measure latency including the probe's cost
  (one GPU→CPU sync per request).

**Second approach: Wanda weight pruning (§12).** Magnitude → Wanda with WikiText statistics → Wanda with decision-input
statistics → decision statistics weighted toward branch tokens (β) and near-tie inputs (margin weighting), at 2:4 and unstructured
50%. Statistics come from the probe-training set, never the sweep. Wanda changes weights, not depth, so at batch 1 it is a
quality and memory method; §12 measures its latency with dense kernels and with real 2:4 sparse kernels. §13 puts every approach in
one table.

Amit estimated about 1.5 h for his original L4 notebook. He used Drive to keep prediction checkpoints across sessions.

In [1]:
# §0 Config
# Use the Colab VM for all model downloads and computation.
SAVE_TO_DRIVE = False  # Set True to keep prediction checkpoints in your own Drive.
DRIVE_ROOT = "/content/ese6800_omnijev"
RUN_NAME = "tien_dee_layer_selection"
QUICK       = False
N_SWEEP     = 100 if QUICK else 500
N_CALIB     = 64            # kept only so the splits match Iterations 2-4
N_TRAIN     = 200 if QUICK else 600   # probe-training questions per dataset
SEED        = 0
L_PROBE     = 8             # the probe reads the prefix after this many layers (output of layer L_PROBE - 1)
FLIP_BUDGET = 3.0           # % decisions allowed to differ from the deep path, used to pick the threshold on training data
N_LAT       = 40
PAIRS = {   # shallow path = deep path + these skips. "mask" = FastJEV's own 6-layer mask
    "dense | p8/b8":         {"engine": "dense", "P": [9, 14, 16, 17, 18, 19, 20, 22], "B": [9, 14, 16, 17, 18, 19, 20, 22]},
    "dense | p6/b6":         {"engine": "dense", "P": [16, 17, 18, 19, 20, 22],         "B": [16, 17, 18, 19, 20, 22]},
    "FastJEV 6 | +prefix 6": {"engine": "fj6",   "P": "mask",                           "B": []},
}

import os, json, subprocess, sys, platform, time, math, random, re, base64, contextlib, gc
from google import colab  # This setup requires a Google Colab runtime.
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/ese6800_omnijev"
OUT = f"{DRIVE_ROOT}/results/{RUN_NAME}"; os.makedirs(OUT, exist_ok=True)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu, "| Python", platform.python_version())
assert gpu, "Choose a GPU in Colab before running the notebook."
json.dump({"gpu": gpu, "started": time.ctime(), "config": {"N_SWEEP": N_SWEEP, "N_TRAIN": N_TRAIN, "L_PROBE": L_PROBE,
           "FLIP_BUDGET": FLIP_BUDGET, "PAIRS": PAIRS}}, open(f"{OUT}/env.json", "w"), indent=1)

GPU: Tesla T4, 15360 MiB, 580.82.07, 7.5 | Python 3.13.16


## §1 Install the Colab dependencies

Run the install cell once per new VM. If Colab already imported a different PyTorch version, restart the **kernel**, rerun §0, and continue with the imports cell below. With the default settings, the model, datasets and prediction checkpoints live on Google’s VM. To keep checkpoints when Colab replaces the VM, set `SAVE_TO_DRIVE = True` in §0 before starting. The T4 uses FP16 for the backbone. Amit’s L4 uses BF16 and the chunked attention kernel. On T4, the setup uses the equivalent recurrent kernel for the image and question because the chunked kernel fails to compile.


In [2]:
%cd /content
!test -d FastJEV || git clone -q https://github.com/JieMaMagic/FastJEV
!git -C /content/FastJEV checkout -q 638f4eb09eb8f44a7f1e8fcadc666e2ab945c205
!pip -q uninstall -y torchaudio torchao
!pip -q install --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 triton==3.3.1
!pip -q install "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" "safetensors==0.8.0" "fla-core==0.5.2" \
    "numpy>=2.1,<2.5" "pillow>=11" datasets "huggingface_hub[hf_xet]" matplotlib scikit-learn
!pip -q install --no-deps -e /content/FastJEV

/content


  Installing build dependencies ... 

done
  Checking if build backend supports build_editable ... 

done
  Getting requirements to build editable ... 

done
  Preparing editable metadata (pyproject.toml) ... 

done
  Building editable for fastjev-inference (pyproject.toml) ... 

done


In [3]:
import importlib
want = {"torch": "2.7.1", "transformers": "5.17.0", "peft": "0.21.0"}
for mod, v in want.items():
    got = importlib.import_module(mod).__version__.split("+")[0]
    assert got == v, f"{mod} {got} != {v}: rerun §1 and restart"
import torch, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import log_loss, roc_auc_score
from fla.ops import chunk_gated_delta_rule
sys.path.insert(0, "/content/FastJEV")
# The pinned FastJEV loader assumes every CUDA GPU supports BF16.
# The free T4 supports FP16 natively. Apply this change only to this VM's copy.
from pathlib import Path
if torch.cuda.get_device_capability()[0] < 8:
    loader = Path("/content/FastJEV/mso/infer.py")
    source = loader.read_text()
    assert "else torch.bfloat16" in source or "else torch.float16" in source
    loader.write_text(source.replace("else torch.bfloat16", "else torch.float16"))
    # Both kernels compute the same gated delta rule. Only the recurrent one runs on T4.
    from mso.fast_kernels import enable_fla
    from transformers.models.qwen3_5 import modeling_qwen3_5 as qwen
    assert enable_fla(verbose=False)
    qwen.torch_chunk_gated_delta_rule = qwen.torch_recurrent_gated_delta_rule
from fastjev import FastJEV
from huggingface_hub import snapshot_download
cfg = json.load(open("/content/FastJEV/fastjev/configs/0.8B.json"))
CKPT = f"{DRIVE_ROOT}/weights/OmniJev-0.8B"; BASE = f"{DRIVE_ROOT}/weights/Qwen3.5-0.8B"
for art, path in (("adapter", CKPT), ("base", BASE)):
    a = cfg["artifacts"][art]; snapshot_download(a["repo"], revision=a["revision"], local_dir=path, allow_patterns=list(a["files"]))
print("OK  stack + weights | GPU", torch.cuda.get_device_name())

OK  stack + weights | GPU Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## §2 Data: sweep set (as before) and a disjoint probe-training set
The sweep and calibration splits are identical to Iterations 2–4. Probe-training questions come from what is left:
AI2D items outside the sweep, calibration and Iteration 3's timing probe; POPE questions whose **image** is in neither the sweep
nor the calibration set, so the probe never sees a sweep image.

In [4]:
from datasets import load_dataset
IMG_DIR = "/content/imgs"; os.makedirs(IMG_DIR, exist_ok=True)
LETTERS = "ABCDEFGH"
AI2D = load_dataset("lmms-lab/ai2d", split="test")
POPE = load_dataset("lmms-lab/POPE", split="test")
AI2D_Q    = AI2D.select_columns(["question"])["question"]
AI2D_OPTS = AI2D.select_columns(["options"])["options"]
AI2D_ANS  = AI2D.select_columns(["answer"])["answer"]
POPE_SRC  = POPE.select_columns(["image_source"])["image_source"]
POPE_CAT  = POPE.select_columns(["category"])["category"]

# Verify each cached PNG once. A disconnected session can leave an unfinished image.
from PIL import Image
checked_images = set()

def cached_png(path, dataset, row):
    if path in checked_images:
        return path
    try:
        with Image.open(path) as image:
            image.load()
    except (OSError, SyntaxError):
        if os.path.exists(path):
            print("Rebuilding incomplete image:", os.path.basename(path))
        temporary = f"{path}.{os.getpid()}.tmp"
        dataset[row]["image"].convert("RGB").save(temporary, format="PNG")
        os.replace(temporary, path)
    checked_images.add(path)
    return path

def ai2d_img(i):
    return cached_png(f"{IMG_DIR}/ai2d-{i}.png", AI2D, i)

def pope_img(i):
    return cached_png(f"{IMG_DIR}/pope-{POPE_SRC[i]}.png", POPE, i)

def pope_statement(q):
    m = re.match(r"(?i)^is there (an?|any|some)?\s*(.+?)\s+in the (image|picture|photo)\?$", q.strip())
    if m: return f"There is {(m.group(1) or 'a').lower()} {m.group(2)} in the {m.group(3).lower()}."
    return q.strip()

def ai2d_item(i):
    opts = list(AI2D_OPTS[i]); listing = "\n".join(f"{LETTERS[j]}. {o}" for j, o in enumerate(opts))
    return {"id": f"ai2d-{i}", "gold": LETTERS[int(AI2D_ANS[i])], "type": "choice",
            "request": {"images": [ai2d_img(i)],
                        "question": {"type": "choice", "instructions": f"{AI2D_Q[i]}\n{listing}",
                                     "options": [{"key": LETTERS[j], "text": o} for j, o in enumerate(opts)]}}}

def pope_item(i):
    r = POPE[i]
    return {"id": f"pope-{i}", "gold": r["answer"].strip().lower(), "type": "noul", "category": POPE_CAT[i],
            "request": {"images": [pope_img(i)],
                        "question": {"type": "noul", "instructions": pope_statement(r["question"])}}}

rng = random.Random(SEED)
a_idx = list(range(len(AI2D))); rng.shuffle(a_idx)
AI2D_CALIB, AI2D_REST = a_idx[:N_CALIB], a_idx[N_CALIB:]
imgs = sorted(set(POPE_SRC)); rng.shuffle(imgs)
calib_imgs = set(imgs[:8])
p_cal = [i for i in range(len(POPE)) if POPE_SRC[i] in calib_imgs]; rng.shuffle(p_cal)
POPE_REST = [i for i in range(len(POPE)) if POPE_SRC[i] not in calib_imgs]; rng.shuffle(POPE_REST)
SETS = {"ai2d": [ai2d_item(i) for i in AI2D_REST[:N_SWEEP]],
        "pope": [pope_item(i) for i in POPE_REST[:N_SWEEP]]}

sweep_imgs = {POPE_SRC[i] for i in POPE_REST[:N_SWEEP]}
pope_train = [i for i in POPE_REST[N_SWEEP:] if POPE_SRC[i] not in sweep_imgs][:N_TRAIN]
ai2d_train = AI2D_REST[N_SWEEP + 40:N_SWEEP + 40 + N_TRAIN]
TRAIN = {"ai2d": [ai2d_item(i) for i in ai2d_train], "pope": [pope_item(i) for i in pope_train]}
assert not {it["id"] for v in TRAIN.values() for it in v} & {it["id"] for v in SETS.values() for it in v}
print("sweep", {k: len(v) for k, v in SETS.items()}, "| probe training", {k: len(v) for k, v in TRAIN.items()},
      "| POPE training images", len({POPE_SRC[i] for i in pope_train}), "(none shared with the sweep)")

Rebuilding incomplete image: ai2d-387.png


sweep {'ai2d': 500, 'pope': 500} | probe training {'ai2d': 600, 'pope': 600} | POPE training images 178 (none shared with the sweep)


## §3 Metrics and evaluation helpers

In [5]:
def option_probs(ans, kind):
    p = dict(ans["probabilities"])
    if not p or not all(math.isfinite(x) and x >= 0 for x in p.values()):
        raise ValueError(f"Invalid model probabilities: {p}")
    if kind == "choice":
        abst = p.pop("__abstain__", 0.0); s = sum(p.values()) or 1.0
        return {k: v / s for k, v in p.items()}, abst
    return p, 0.0

def ece(conf, correct, bins=15):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    b = np.clip(np.ceil(conf * bins).astype(int) - 1, 0, bins - 1)
    e = 0.0
    for k in range(bins):
        m = b == k
        if m.any(): e += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return e

def pred(p): return max(p, key=p.get)

def margin(p):
    v = sorted(p.values(), reverse=True)
    return v[0] - (v[1] if len(v) > 1 else 0.0)

def arrays(rows):
    conf = np.array([max(r["p"].values()) for r in rows])
    corr = np.array([pred(r["p"]) == r["gold"] for r in rows], float)
    return conf, corr

def summarize(rows, ref=None, boot=True):
    rows = list(rows); conf, corr = arrays(rows)
    s = {"n": len(rows), "acc": 100 * corr.mean(), "ece": 100 * ece(conf, corr),
         "brier": float(np.mean([sum((v - (k == r["gold"])) ** 2 for k, v in r["p"].items()) for r in rows])),
         "nll": float(np.mean([-math.log(max(r["p"].get(r["gold"], 0.0), 1e-12)) for r in rows]))}
    if ref is not None:
        rr = [ref[r["id"]] for r in rows]; rk = arrays(rr)[1]
        s["flip"] = 100 * np.mean([pred(a["p"]) != pred(b["p"]) for a, b in zip(rows, rr)])
        s["d_acc"] = s["acc"] - 100 * rk.mean()
        if boot:
            g = np.random.default_rng(SEED)
            da = [100 * (corr[ix].mean() - rk[ix].mean()) for ix in (g.integers(0, len(rows), len(rows)) for _ in range(1000))]
            s["d_acc_ci"] = [float(np.percentile(da, 2.5)), float(np.percentile(da, 97.5))]
    return s

def temper(p, T):
    keys = list(p); v = np.array([max(p[k], 1e-12) for k in keys]) ** (1.0 / T); v = v / v.sum()
    return dict(zip(keys, map(float, v)))

def fit_T(rows):
    grid = np.exp(np.linspace(math.log(0.25), math.log(4.0), 121))
    nll = [np.mean([-math.log(max(temper(r["p"], T)[r["gold"]], 1e-12)) for r in rows]) for T in grid]
    return float(grid[int(np.argmin(nll))])

def crossfit_ece(rows, k=5):
    ids = list(rows); random.Random(SEED).shuffle(ids); out = []
    for f in range(k):
        test = set(ids[f::k]); T = fit_T([rows[i] for i in ids if i not in test])
        out += [{**rows[i], "p": temper(rows[i]["p"], T)} for i in test]
    return summarize(out)["ece"]

def aurc(rows):
    conf, corr = arrays(list(rows)); o = np.argsort(-conf); err = 1 - corr[o]
    return float(100 * (np.cumsum(err) / np.arange(1, len(err) + 1)).mean())

CUR = {}
def ask(engine, it):
    CUR.update(calls={}, shallow=False, feat=None, score=None)
    return engine.predict(it["request"], image_root="/")["answers"]["q"]

def run_eval(engine, tag, items):
    # Resumable JSONL. Rows also store the probe feature (when captured) and the routing decision (when adaptive)
    path = f"{OUT}/{tag}.jsonl"; have = {}
    if os.path.exists(path):
        for l in open(path):
            r = json.loads(l); have[r["id"]] = r
    with open(path, "a") as f:
        for k, it in enumerate(items):
            if it["id"] in have: continue
            torch.cuda.synchronize(); t = time.perf_counter()
            ans = ask(engine, it)
            torch.cuda.synchronize(); dt = 1000 * (time.perf_counter() - t)
            p, abst = option_probs(ans, it["type"])
            r = {"id": it["id"], "gold": it["gold"], "p": p, "ms": dt, "shallow": bool(CUR["shallow"]), "score": CUR["score"]}
            if CUR["feat"] is not None: r["feat"] = base64.b64encode(CUR["feat"].tobytes()).decode()
            have[it["id"]] = r; f.write(json.dumps(r) + "\n")
            if k % 100 == 0: f.flush()
    return {it["id"]: have[it["id"]] for it in items}

def feats(rows):
    return np.stack([np.frombuffer(base64.b64decode(r["feat"]), np.float16).astype(np.float32) for r in rows])

RESULTS = {}
def save(): json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=float)

## §4 Engines and the routing hook
`Route(engine, P, B, mode, probe, thr, capture)`:
* `mode="deep"`: no extra skipping. `"shallow"`: always skip P in the prefix and B in the branches. `"adaptive"`: the probe decides
  per request.
* The probe hook sits on the output of layer `L_PROBE − 1` during its **first call** (the prefix). It builds the feature
  [last-token state, mean state] (2 × hidden size), and in adaptive mode computes `sigmoid(w·x + b)` and sets the route.
* Every layer in P ∪ B must come **after** the probe layer, so the route is known before any of them runs. Nesting
  (P ⊆ B ∪ FastJEV mask) is enforced.

In [6]:
def lm_layers(engine):
    mods = dict(engine.model.model.backbone.named_modules())
    return mods[min((n for n in mods if n.endswith("language_model")), key=len)].layers

ENG, FJ_MASK = {}, {}
ENG["dense"] = FastJEV(CKPT, BASE, scale="0.8B", mode="original", protocol="public")
if any(p["engine"] == "fj6" for p in PAIRS.values()):
    ENG["fj6"] = FastJEV(CKPT, BASE, scale="0.8B", mode="independent", pruned_layers=6, protocol="public")
    FJ_MASK["fj6"] = sorted(getattr(ENG["fj6"], "layers", cfg["pruning_masks"]["6"]))
for e in ENG.values(): e.model.model.backbone.merge_adapter()
LAYERS = {n: lm_layers(e) for n, e in ENG.items()}
NL = len(LAYERS["dense"]); TYPES = [l.block_type for l in LAYERS["dense"]]; FIRST_FULL = TYPES.index("full_attention")
for key, p in PAIRS.items():
    if p["P"] == "mask": p["P"] = FJ_MASK[p["engine"]]
    bad = [l for l in set(p["P"]) | set(p["B"]) if l < L_PROBE or l in (0, NL - 1, FIRST_FULL)]
    assert not bad, f"{key}: layers {bad} are before the probe layer or never skippable"
print(NL, "layers | FastJEV mask:", FJ_MASK, "| pairs:", {k: (p["P"], p["B"]) for k, p in PAIRS.items()})

class Route:
    def __init__(self, eng, P=(), B=(), mode="deep", probe=None, thr=None, capture=False):
        self.L, self.P, self.B = LAYERS[eng], set(P), set(B)
        assert self.P <= self.B | set(FJ_MASK.get(eng, [])), "prefix-skipped layers must also be skipped in the branches"
        self.mode, self.probe, self.thr, self.capture = mode, probe, thr, capture
    def __enter__(self):
        key, self.handles = id(self.L), []
        if self.capture or self.mode == "adaptive":
            def hook(m, a, k, out):
                n = CUR["calls"].get((key, "probe"), 0); CUR["calls"][(key, "probe")] = n + 1
                if n: return
                h = out[0] if isinstance(out, (tuple, list)) else out
                x = torch.cat([h[0, -1], h[0].mean(0)]).float()
                if self.capture: CUR["feat"] = x.detach().cpu().numpy().astype(np.float16)
                if self.mode == "adaptive":
                    s = torch.sigmoid(x @ self.probe[0] + self.probe[1]).item()   # one GPU->CPU sync per request
                    CUR["score"], CUR["shallow"] = s, s >= self.thr
            self.handles.append(self.L[L_PROBE - 1].register_forward_hook(hook, with_kwargs=True))
        if self.mode != "deep":
            for i in self.P | self.B:
                orig = self.L[i].forward
                def fwd(hidden_states, *a, _orig=orig, _i=i, **k):
                    n = CUR["calls"].get((key, _i), 0); CUR["calls"][(key, _i)] = n + 1
                    on = self.mode == "shallow" or CUR["shallow"]
                    if on and ((n == 0 and _i in self.P) or (n > 0 and _i in self.B)):
                        return hidden_states
                    return _orig(hidden_states, *a, **k)
                self.L[i].forward = fwd
        return self
    def __exit__(self, *exc):
        for h in self.handles: h.remove()
        for i in self.P | self.B: self.L[i].__dict__.pop("forward", None)
        return False

# Use a practice question to check that the first call processes the input prefix.
for name in ENG:
    shapes = []
    h = LAYERS[name][L_PROBE - 1].register_forward_pre_hook(
        lambda m, a, k: shapes.append(tuple((a[0] if a else k["hidden_states"]).shape[:2])), with_kwargs=True)
    with torch.inference_mode(): ask(ENG[name], TRAIN["ai2d"][0])
    h.remove()
    print(f"{name}: calls to layer {L_PROBE - 1} (batch, tokens): {shapes}")
    assert shapes and shapes[0][0] == 1 and shapes[0][1] == max(s[1] for s in shapes) and shapes[0][1] > 32, \
        "the first call is not the prefix: routing would be wrong"

24 layers | FastJEV mask: {'fj6': [15, 16, 17, 18, 19, 20]} | pairs: {'dense | p8/b8': ([9, 14, 16, 17, 18, 19, 20, 22], [9, 14, 16, 17, 18, 19, 20, 22]), 'dense | p6/b6': ([16, 17, 18, 19, 20, 22], [16, 17, 18, 19, 20, 22]), 'FastJEV 6 | +prefix 6': ([15, 16, 17, 18, 19, 20], [])}


dense: calls to layer 7 (batch, tokens): [(1, 285), (4, 7)]


fj6: calls to layer 7 (batch, tokens): [(1, 285), (4, 7)]


In [13]:
# Re-run model and routing initialization
def lm_layers(engine):
    mods = dict(engine.model.model.backbone.named_modules())
    return mods[min((n for n in mods if n.endswith("language_model")), key=len)].layers

ENG, FJ_MASK = {}, {}
ENG["dense"] = FastJEV(CKPT, BASE, scale="0.8B", mode="original", protocol="public")
if any(p["engine"] == "fj6" for p in PAIRS.values()):
    ENG["fj6"] = FastJEV(CKPT, BASE, scale="0.8B", mode="independent", pruned_layers=6, protocol="public")
    FJ_MASK["fj6"] = sorted(getattr(ENG["fj6"], "layers", cfg["pruning_masks"]["6"]))
for e in ENG.values(): e.model.model.backbone.merge_adapter()
LAYERS = {n: lm_layers(e) for n, e in ENG.items()}
NL = len(LAYERS["dense"]); TYPES = [l.block_type for l in LAYERS["dense"]]; FIRST_FULL = TYPES.index("full_attention")
for key, p in PAIRS.items():
    if p["P"] == "mask": p["P"] = FJ_MASK[p["engine"]]
    bad = [l for l in set(p["P"]) | set(p["B"]) if l < L_PROBE or l in (0, NL - 1, FIRST_FULL)]
    assert not bad, f"{key}: layers {bad} are before the probe layer or never skippable"
print(NL, "layers | FastJEV mask:", FJ_MASK, "| pairs:", {k: (p["P"], p["B"]) for k, p in PAIRS.items()})

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

24 layers | FastJEV mask: {'fj6': [15, 16, 17, 18, 19, 20]} | pairs: {'dense | p8/b8': ([9, 14, 16, 17, 18, 19, 20, 22], [9, 14, 16, 17, 18, 19, 20, 22]), 'dense | p6/b6': ([16, 17, 18, 19, 20, 22], [16, 17, 18, 19, 20, 22]), 'FastJEV 6 | +prefix 6': ([15, 16, 17, 18, 19, 20], [])}


## §5 Deep and shallow runs (training and sweep sets)
Deep runs capture the probe feature. Shallow runs are the static shallow path for each pair. ≈ 30 min in total.

In [14]:
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
ROWS = {}
for eng in sorted({p["engine"] for p in PAIRS.values()} | {"dense"}):
    with Route(eng, mode="deep", capture=True):
        for split, sets in (("train", TRAIN), ("sweep", SETS)):
            for ds, items in sets.items():
                ROWS[("deep", eng, split, ds)] = run_eval(ENG[eng], f"deep_{eng}_{split}_{ds}", items)
    print("deep", eng, "done")
for key, p in PAIRS.items():
    with Route(p["engine"], p["P"], p["B"], mode="shallow"):
        for split, sets in (("train", TRAIN), ("sweep", SETS)):
            for ds, items in sets.items():
                ROWS[("shallow", key, split, ds)] = run_eval(ENG[p["engine"]], f"shallow_{slug(key)}_{split}_{ds}", items)
    for split in ("train", "sweep"):
        fl = {ds: 100 * np.mean([pred(ROWS[("shallow", key, split, ds)][i]["p"]) != pred(ROWS[("deep", p["engine"], split, ds)][i]["p"])
                                 for i in ROWS[("deep", p["engine"], split, ds)]]) for ds in SETS}
        print(f"{key:24s} {split:5s} flips vs deep: AI2D {fl['ai2d']:.1f}%  POPE {fl['pope']:.1f}%")

deep dense done
deep fj6 done
dense | p8/b8            train flips vs deep: AI2D 28.0%  POPE 7.8%
dense | p8/b8            sweep flips vs deep: AI2D 26.6%  POPE 9.0%
dense | p6/b6            train flips vs deep: AI2D 5.2%  POPE 4.5%
dense | p6/b6            sweep flips vs deep: AI2D 7.6%  POPE 3.6%
FastJEV 6 | +prefix 6    train flips vs deep: AI2D 4.5%  POPE 0.5%
FastJEV 6 | +prefix 6    sweep flips vs deep: AI2D 3.8%  POPE 0.8%


## §6 Train the probes (training set only)
Target: `safe = 1` if the shallow path picks the same option as the deep path. One probe per pair, shared across AI2D and POPE.
Regularization C is chosen by 5-fold cross-validated log loss. The threshold is the lowest one whose **out-of-fold** flip rate
stays under `FLIP_BUDGET`, i.e. the most inputs routed shallow within the budget. Nothing here looks at the sweep set.

In [15]:
def pick_threshold(score, safe, budget):
    for t in np.unique(np.concatenate([score, [1.01]])):
        if 100 * np.mean((score >= t) & (safe == 0)) <= budget: return float(t)
    return 1.01

PROBES = {}
for key, p in PAIRS.items():
    X, y = [], []
    for ds in TRAIN:
        d, s = ROWS[("deep", p["engine"], "train", ds)], ROWS[("shallow", key, "train", ds)]
        ids = list(d); X.append(feats([d[i] for i in ids])); y += [int(pred(s[i]["p"]) == pred(d[i]["p"])) for i in ids]
    X, y = np.concatenate(X), np.array(y)
    mu, sd = X.mean(0), X.std(0) + 1e-4; Z = (X - mu) / sd
    if y.min() == 1:
        print(key, ": shallow path never flips on training data, routing everything shallow")
        PROBES[key] = {"w": np.zeros(X.shape[1]), "b": 10.0, "thr": 0.0, "auc": float("nan"), "safe_rate": 1.0}; continue
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED); best = None
    for C in (0.0003, 0.001, 0.003, 0.01, 0.03, 0.1):
        oof = cross_val_predict(LogisticRegression(C=C, max_iter=3000), Z, y, cv=cv, method="predict_proba")[:, 1]
        if best is None or log_loss(y, oof) < best[0]: best = (log_loss(y, oof), C, oof)
    _, C, oof = best
    clf = LogisticRegression(C=C, max_iter=3000).fit(Z, y)
    w = clf.coef_[0] / sd; b = float(clf.intercept_[0] - (mu / sd) @ clf.coef_[0])
    thr = pick_threshold(oof, y, FLIP_BUDGET)
    PROBES[key] = {"w": w, "b": b, "thr": thr, "C": C, "auc": float(roc_auc_score(y, oof)), "safe_rate": float(y.mean()),
                   "oof_shallow": float(np.mean(oof >= thr)), "oof_flip": float(100 * np.mean((oof >= thr) & (y == 0)))}
    q = PROBES[key]
    print(f"{key:24s} safe {100 * q['safe_rate']:.1f}% | probe AUC {q['auc']:.3f} (C={C}) | threshold {thr:.3f} -> "
          f"out-of-fold: {100 * q['oof_shallow']:.1f}% routed shallow, {q['oof_flip']:.1f}% flips")
RESULTS["_probes"] = {k: {kk: v for kk, v in q.items() if kk != "w"} for k, q in PROBES.items()}; save()
TORCH_PROBE = {k: (torch.tensor(q["w"], dtype=torch.float32, device="cuda"), float(q["b"])) for k, q in PROBES.items()}

dense | p8/b8            safe 82.1% | probe AUC 0.725 (C=0.001) | threshold 0.822 -> out-of-fold: 50.1% routed shallow, 3.0% flips
dense | p6/b6            safe 95.2% | probe AUC 0.674 (C=0.003) | threshold 0.906 -> out-of-fold: 88.5% routed shallow, 3.0% flips
FastJEV 6 | +prefix 6    safe 97.5% | probe AUC 0.859 (C=0.003) | threshold 0.355 -> out-of-fold: 100.0% routed shallow, 2.5% flips


## §7 Run the adaptive path for real (sweep set)
Each pair is run end to end with the probe deciding per request. The routing decisions are compared with the offline
prediction from the captured features: they should agree on ≥ 99% of requests (small differences can come from BF16).

In [16]:
ADAPT = {}
for key, p in PAIRS.items():
    with Route(p["engine"], p["P"], p["B"], mode="adaptive", probe=TORCH_PROBE[key], thr=PROBES[key]["thr"]):
        ADAPT[key] = {ds: run_eval(ENG[p["engine"]], f"adaptive_{slug(key)}_{ds}", items) for ds, items in SETS.items()}
    for ds in SETS:
        d = ROWS[("deep", p["engine"], "sweep", ds)]; ids = list(d)
        off = 1 / (1 + np.exp(-(feats([d[i] for i in ids]) @ PROBES[key]["w"] + PROBES[key]["b"]))) >= PROBES[key]["thr"]
        on = np.array([ADAPT[key][ds][i]["shallow"] for i in ids])
        print(f"{key:24s} {ds:5s} routed shallow {100 * on.mean():5.1f}% | online vs offline routing agree {100 * np.mean(on == off):.1f}%")

dense | p8/b8            ai2d  routed shallow  12.0% | online vs offline routing agree 100.0%
dense | p8/b8            pope  routed shallow  91.2% | online vs offline routing agree 100.0%
dense | p6/b6            ai2d  routed shallow  91.4% | online vs offline routing agree 100.0%
dense | p6/b6            pope  routed shallow  90.4% | online vs offline routing agree 100.0%
FastJEV 6 | +prefix 6    ai2d  routed shallow 100.0% | online vs offline routing agree 100.0%
FastJEV 6 | +prefix 6    pope  routed shallow 100.0% | online vs offline routing agree 100.0%


## §8 Same-session latency
Interleaved per item. "probe only" runs the probe but never takes the shallow path, which isolates the probe's own cost.

In [17]:
SPECS = {"dense": dict(eng="dense")}
if "fj6" in ENG: SPECS["FastJEV S+P 6"] = dict(eng="fj6")
for key, p in PAIRS.items():
    base = dict(eng=p["engine"], P=p["P"], B=p["B"])
    SPECS[f"{key} [static shallow]"] = dict(base, mode="shallow")
    SPECS[f"{key} [adaptive]"] = dict(base, mode="adaptive", probe=TORCH_PROBE[key], thr=PROBES[key]["thr"])
    SPECS[f"{key} [probe only]"] = dict(base, mode="adaptive", probe=TORCH_PROBE[key], thr=2.0)

def time_one(spec, it):
    with Route(**spec):
        torch.cuda.synchronize(); t = time.perf_counter(); ask(ENG[spec["eng"]], it); torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t)

names = list(SPECS); LAT = {n: {ds: [] for ds in SETS} for n in names}
for ds in SETS:
    for k, it in enumerate(SETS[ds][:N_LAT + 3]):
        for n in names[k % len(names):] + names[:k % len(names)]:
            dt = time_one(SPECS[n], it)
            if k >= 3: LAT[n][ds].append(dt)
MED = {n: {ds: float(np.median(LAT[n][ds])) for ds in SETS} for n in names}
SPEED = {n: {ds: MED["dense"][ds] / MED[n][ds] for ds in SETS} for n in names}
RESULTS["_latency_ms"], RESULTS["_speedup"] = MED, SPEED; save()
for n in names:
    print(f"{n:40s} AI2D {MED[n]['ai2d']:6.1f} ms ({SPEED[n]['ai2d']:.3f}x) | POPE {MED[n]['pope']:6.1f} ms ({SPEED[n]['pope']:.3f}x)")
for key, p in PAIRS.items():
    deep = "dense" if p["engine"] == "dense" else "FastJEV S+P 6"
    print(f"probe cost for {key}: {np.mean([MED[f'{key} [probe only]'][ds] - MED[deep][ds] for ds in SETS]):+.1f} ms per request")

dense                                    AI2D  187.0 ms (1.000x) | POPE  180.9 ms (1.000x)
FastJEV S+P 6                            AI2D  156.3 ms (1.196x) | POPE  150.0 ms (1.206x)
dense | p8/b8 [static shallow]           AI2D  139.3 ms (1.343x) | POPE  134.3 ms (1.347x)
dense | p8/b8 [adaptive]                 AI2D  184.1 ms (1.016x) | POPE  134.4 ms (1.346x)
dense | p8/b8 [probe only]               AI2D  186.0 ms (1.005x) | POPE  179.8 ms (1.006x)
dense | p6/b6 [static shallow]           AI2D  152.6 ms (1.225x) | POPE  145.0 ms (1.248x)
dense | p6/b6 [adaptive]                 AI2D  152.4 ms (1.227x) | POPE  146.2 ms (1.237x)
dense | p6/b6 [probe only]               AI2D  186.4 ms (1.003x) | POPE  179.9 ms (1.006x)
FastJEV 6 | +prefix 6 [static shallow]   AI2D  139.4 ms (1.341x) | POPE  132.5 ms (1.366x)
FastJEV 6 | +prefix 6 [adaptive]         AI2D  138.8 ms (1.348x) | POPE  133.5 ms (1.355x)
FastJEV 6 | +prefix 6 [probe only]       AI2D  155.5 ms (1.203x) | POPE  149.0 ms (1.214x)

## §9 Results: does the probe beat random routing, and does adaptive depth beat FastJEV?
For each pair and dataset:
* **adaptive** = the real run from §7 at the threshold chosen on training data.
* **random** = random routing with the same shallow fraction (expected accuracy and flips, closed form).
* **oracle** = shallow exactly when it is safe (upper bound for any router).
* Δ vs FastJEV is a paired bootstrap on the same questions.

In [18]:
FJ_ROWS = {ds: ROWS[("deep", "fj6", "sweep", ds)] for ds in SETS} if "fj6" in ENG else None
DENSE = {ds: ROWS[("deep", "dense", "sweep", ds)] for ds in SETS}
TAB, lines = {}, []
for key, p in PAIRS.items():
    deep_name = "dense" if p["engine"] == "dense" else "FastJEV S+P 6"
    for ds in SETS:
        d, s, a = ROWS[("deep", p["engine"], "sweep", ds)], ROWS[("shallow", key, "sweep", ds)], ADAPT[key][ds]
        ids = list(d); safe = np.array([pred(s[i]["p"]) == pred(d[i]["p"]) for i in ids])
        f = float(np.mean([a[i]["shallow"] for i in ids]))
        acc = lambda rows: 100 * np.mean([pred(rows[i]["p"]) == rows[i]["gold"] for i in ids])
        sa = summarize(a.values(), DENSE[ds]); fl = summarize(a.values(), d, boot=False)["flip"]
        row = {"shallow_frac": 100 * f, "acc_deep": acc(d), "acc_shallow": acc(s), "acc_adaptive": sa["acc"],
               "acc_random": (1 - f) * acc(d) + f * acc(s), "flip_adaptive": fl, "flip_random": 100 * f * (1 - safe.mean()),
               "oracle_frac": 100 * safe.mean(),
               "acc_oracle": 100 * np.mean([pred((s if ok else d)[i]["p"]) == d[i]["gold"] for i, ok in zip(ids, safe)]),
               "d_dense": sa["d_acc"], "d_dense_ci": sa["d_acc_ci"],
               "speed": SPEED[f"{key} [adaptive]"][ds], "speed_deep": SPEED[deep_name][ds],
               "speed_shallow": SPEED[f"{key} [static shallow]"][ds]}
        if FJ_ROWS:
            v = summarize(a.values(), FJ_ROWS[ds]); row["d_fj"], row["d_fj_ci"] = v["d_acc"], v["d_acc_ci"]
        TAB.setdefault(key, {})[ds] = row
        r = row
        lines.append(f"| {key} | {ds.upper()} | {r['shallow_frac']:.0f}% | {r['acc_deep']:.1f} / {r['acc_shallow']:.1f} | "
                     f"**{r['acc_adaptive']:.1f}** | {r['acc_random']:.1f} | {r['acc_oracle']:.1f} ({r['oracle_frac']:.0f}%) | "
                     f"{r['flip_adaptive']:.1f} / {r['flip_random']:.1f} | {r['speed_deep']:.2f} / {r['speed_shallow']:.2f} / **{r['speed']:.2f}** | "
                     f"{r['d_dense']:+.1f} [{r['d_dense_ci'][0]:+.1f}, {r['d_dense_ci'][1]:+.1f}] | "
                     + (f"{r['d_fj']:+.1f} [{r['d_fj_ci'][0]:+.1f}, {r['d_fj_ci'][1]:+.1f}] |" if "d_fj" in r else "– |"))
RESULTS["routing"] = TAB; save()
hdr = ("| Pair | Set | Routed shallow | Acc deep / shallow | Acc adaptive | Acc random routing | Acc oracle (frac) | "
       "Flips adaptive / random % | Speedup deep / shallow / adaptive | Δ vs dense | Δ vs FastJEV |\n" + "|---" * 11 + "|\n")
display(Markdown(hdr + "\n".join(lines))); open(f"{OUT}/routing_table.md", "w").write(hdr + "\n".join(lines))

if FJ_ROWS:
    print("\nVerdict vs FastJEV S+P 6 (faster on both sets, Δacc > -1 and CI lower end > -2 on both):")
    for key in PAIRS:
        faster = all(TAB[key][ds]["speed"] > SPEED["FastJEV S+P 6"][ds] for ds in SETS)
        holds = all(TAB[key][ds]["d_fj"] > -1 and TAB[key][ds]["d_fj_ci"][0] > -2 for ds in SETS)
        print(f"  {'BEATS' if faster and holds else 'trades' if faster else 'slower' if holds else 'worse':6s} {key}")

| Pair | Set | Routed shallow | Acc deep / shallow | Acc adaptive | Acc random routing | Acc oracle (frac) | Flips adaptive / random % | Speedup deep / shallow / adaptive | Δ vs dense | Δ vs FastJEV |
|---|---|---|---|---|---|---|---|---|---|---|
| dense | p8/b8 | AI2D | 12% | 60.8 / 58.4 | **59.8** | 60.5 | 60.8 (73%) | 3.0 / 3.2 | 1.00 / 1.34 / **1.02** | -1.0 [-2.4, +0.4] | -2.0 [-4.8, +0.4] |
| dense | p8/b8 | POPE | 91% | 80.6 / 73.6 | **76.4** | 74.2 | 80.6 (91%) | 6.2 / 8.2 | 1.00 / 1.35 / **1.35** | -4.2 [-6.4, -2.4] | -3.2 [-5.4, -1.2] |
| dense | p6/b6 | AI2D | 91% | 60.8 / 61.4 | **61.4** | 61.3 | 60.8 (92%) | 6.6 / 6.9 | 1.00 / 1.23 / **1.23** | +0.6 [-1.2, +2.4] | -0.4 [-2.6, +1.8] |
| dense | p6/b6 | POPE | 90% | 80.6 / 77.0 | **78.2** | 77.3 | 80.6 (96%) | 2.4 / 3.3 | 1.00 / 1.25 / **1.24** | -2.4 [-4.0, -1.2] | -1.4 [-3.0, +0.0] |
| FastJEV 6 | +prefix 6 | AI2D | 100% | 61.8 / 62.4 | **62.4** | 62.4 | 61.8 (96%) | 3.8 / 3.8 | 1.20 / 1.34 / **1.35** | +1.6 [-0.4, +3.6] | +0.6 [-1.0, +2.0] |
| FastJEV 6 | +prefix 6 | POPE | 100% | 79.6 / 78.8 | **78.8** | 78.8 | 79.6 (99%) | 0.8 / 0.8 | 1.21 / 1.37 / **1.36** | -1.8 [-3.0, -0.6] | -0.8 [-1.8, -0.2] |


Verdict vs FastJEV S+P 6 (faster on both sets, Δacc > -1 and CI lower end > -2 on both):
  worse  dense | p8/b8
  trades dense | p6/b6
  BEATS  FastJEV 6 | +prefix 6


## §10 Accuracy–speed curves
The curve sweeps the threshold on the sweep set (descriptive only; the reported point uses the training-set threshold).
Expected latency = f·t_shallow + (1 − f)·t_deep + probe cost, from §8. Random routing is the straight line between deep and
shallow; the probe is useful where its curve lies above that line.

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, ds in zip(axes, SETS):
    for key, p in PAIRS.items():
        deep_name = "dense" if p["engine"] == "dense" else "FastJEV S+P 6"
        d, s = ROWS[("deep", p["engine"], "sweep", ds)], ROWS[("shallow", key, "sweep", ds)]; ids = list(d)
        sc = 1 / (1 + np.exp(-(feats([d[i] for i in ids]) @ PROBES[key]["w"] + PROBES[key]["b"])))
        td, tsh = MED[deep_name][ds], MED[f"{key} [static shallow]"][ds]
        oh = MED[f"{key} [probe only]"][ds] - td
        xs, ys = [], []
        for t in np.quantile(sc, np.linspace(0, 1, 21)):
            r = sc >= t; f = r.mean()
            xs.append(MED["dense"][ds] / (f * tsh + (1 - f) * td + oh))
            ys.append(100 * np.mean([pred((s if ri else d)[i]["p"]) == d[i]["gold"] for i, ri in zip(ids, r)]))
        line, = ax.plot(xs, ys, marker=".", label=f"{key} (probe)")
        a_d, a_s = TAB[key][ds]["acc_deep"], TAB[key][ds]["acc_shallow"]
        ax.plot([MED["dense"][ds] / td, MED["dense"][ds] / tsh], [a_d, a_s], ls=":", c=line.get_color(), lw=1)
        ax.scatter(TAB[key][ds]["speed"], TAB[key][ds]["acc_adaptive"], c=line.get_color(), marker="D", s=50, zorder=5)
    if "FastJEV S+P 6" in SPEED:
        ax.scatter(SPEED["FastJEV S+P 6"][ds], summarize(FJ_ROWS[ds].values())["acc"], marker="*", s=200, c="red", zorder=6, label="FastJEV S+P 6")
    ax.scatter(1, summarize(DENSE[ds].values())["acc"], marker="s", c="black", zorder=6, label="dense")
    ax.set_xlabel("speedup vs dense"); ax.set_ylabel("accuracy %"); ax.set_title(ds.upper())
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/adaptive_curves.png", dpi=150); plt.show()
print("dotted = random routing, diamond = real adaptive run at the training-set threshold")

dotted = random routing, diamond = real adaptive run at the training-set threshold


## §11 Calibration of the adaptive models
NLL and Brier (proper scoring rules), AURC (does confidence still rank answers?), and ECE after a 5-fold cross-fitted
temperature. Compared with the deep path and with FastJEV.

In [20]:
CAL = {}
rows_by = {"dense": DENSE}
if FJ_ROWS: rows_by["FastJEV S+P 6"] = FJ_ROWS
for key in PAIRS: rows_by[f"{key} [adaptive]"] = ADAPT[key]
lines = []
for n, rr in rows_by.items():
    c = {ds: dict(summarize(rr[ds].values(), boot=False), aurc=aurc(rr[ds].values()), ece_T=crossfit_ece(rr[ds])) for ds in SETS}
    CAL[n] = c
    lines.append(f"| {n} | " + " | ".join(f"{c[ds]['nll']:.3f} / {c[ds]['brier']:.3f} / {c[ds]['aurc']:.1f} / {c[ds]['ece']:.1f} / {c[ds]['ece_T']:.1f}"
                                          for ds in SETS) + " |")
RESULTS["calibration"] = CAL; save()
display(Markdown("| Config | AI2D NLL / Brier / AURC / ECE / ECE(T) | POPE NLL / Brier / AURC / ECE / ECE(T) |\n|---|---|---|\n" + "\n".join(lines)))

| Config | AI2D NLL / Brier / AURC / ECE / ECE(T) | POPE NLL / Brier / AURC / ECE / ECE(T) |
|---|---|---|
| dense | 0.995 / 0.518 / 20.7 / 5.7 / 3.3 | 0.520 / 0.309 / 8.8 / 11.1 / 2.5 |
| FastJEV S+P 6 | 0.971 / 0.507 / 19.4 / 4.6 / 6.8 | 0.472 / 0.302 / 10.6 / 7.2 / 3.7 |
| dense | p8/b8 [adaptive] | 1.017 / 0.531 / 22.4 / 4.7 / 3.0 | 0.577 / 0.357 / 13.0 / 12.3 / 5.9 |
| dense | p6/b6 [adaptive] | 0.956 / 0.504 / 19.5 / 4.1 / 6.3 | 0.497 / 0.315 / 10.8 / 8.0 / 3.6 |
| FastJEV 6 | +prefix 6 [adaptive] | 0.959 / 0.503 / 19.2 / 3.6 / 6.5 | 0.480 / 0.307 / 10.8 / 7.1 / 3.0 |

## §12 Wanda weight pruning
Wanda scores weight W_ij by |W_ij| · ‖X_j‖, the weight times the norm of its input feature, and keeps the top weights in each
group (2 of every 4 for 2:4, the top half of each output row for unstructured 50%). The ladder changes only where ‖X_j‖ comes from:

| Method | Activation statistics |
|---|---|
| `magnitude` | none, |W| only |
| `text` | 128 × 256-token WikiText-2 sequences (the Wanda paper's setting) |
| `dec` | 128 decision inputs from the probe-training set, every token weight 1 |
| `dec_mw` | same inputs, branch tokens weighted β = 4, each input weighted exp(−margin/τ) so near-ties count most |

All linear projections of decoder layers 0–22 are pruned (merged LoRA included); the last layer stays dense. Masks act on dense
weights, so accuracy is measured with ordinary kernels; latency is measured both ways at the end.

In [21]:
WANDA_METHODS  = ["magnitude", "text", "dec", "dec_mw"]
WANDA_PATTERNS = ["2:4"] if QUICK else ["2:4", "50%"]
BETA, TAU      = 4.0, 0.1
N_WANDA_CAL    = 64            # per dataset, taken from the probe-training set (disjoint from the sweep)

dense = ENG["dense"]; DL = LAYERS["dense"]
_m = dict(dense.model.model.backbone.named_modules())
LM = _m[min((n for n in _m if n.endswith("language_model")), key=len)]
TARGETS = {}
for li, layer in enumerate(DL):
    if li == NL - 1: continue
    for n, m in layer.named_modules():
        if isinstance(m, torch.nn.Linear) and "lora_" not in n and m.in_features % 4 == 0 and m.out_features >= 256:
            TARGETS[f"{li}.{n}"] = m
ORIG = {n: m.weight.detach().to("cpu", copy=True) for n, m in TARGETS.items()}
def restore():
    for n, m in TARGETS.items(): m.weight.data.copy_(ORIG[n].to(m.weight.device))
print(len(TARGETS), "projections,", f"{sum(m.weight.numel() for m in TARGETS.values()) / 1e6:.0f}M weights")

# Activation statistics
STATS = {k: {} for k in ("text", "dec", "dec_mw")}
MODE = {"m": None, "w": 1.0}
def _acc(k, n, v): STATS[k][n] = STATS[k][n] + v if n in STATS[k] else v
def stat_hook(name):
    def hook(mod, args):
        x = args[0]; sq = x.detach().float().pow(2).reshape(-1, x.shape[-1]).sum(0)
        if MODE["m"] == "text": _acc("text", name, sq)
        elif MODE["m"] == "dec":
            c = CUR["calls"].get(("wanda", name), 0); CUR["calls"][("wanda", name)] = c + 1
            b = 1.0 if c == 0 else BETA                     # first call of a projection in a request = the prefix
            _acc("dec", name, sq); _acc("dec_mw", name, MODE["w"] * b * sq)
    return hook

WCAL = TRAIN["ai2d"][:N_WANDA_CAL] + TRAIN["pope"][:N_WANDA_CAL]
deep_tr = {**ROWS[("deep", "dense", "train", "ai2d")], **ROWS[("deep", "dense", "train", "pope")]}
wx = np.exp(-np.array([margin(deep_tr[it["id"]]["p"]) for it in WCAL]) / TAU); wx = wx * len(wx) / wx.sum()
handles = [m.register_forward_pre_hook(stat_hook(n)) for n, m in TARGETS.items()]
try:
    MODE["m"] = "dec"
    for it, w in zip(WCAL, wx):
        MODE["w"] = float(w)
        with torch.inference_mode(): ask(dense, it)
    MODE["m"] = "text"
    wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    ids = dense.model.proc.tokenizer("\n\n".join(t for t in wt["text"] if t.strip())[:2_000_000])["input_ids"]
    seqs = torch.tensor([ids[k * 256:(k + 1) * 256] for k in range(128)], device="cuda")
    with torch.inference_mode():
        for k in range(0, 128, 8): LM(input_ids=seqs[k:k + 8], use_cache=False)
finally:
    MODE["m"] = None
    for h in handles: h.remove()
print("statistics collected:", {k: len(v) for k, v in STATS.items()})

def prune(method, pattern):
    restore()
    for n, m in TARGETS.items():
        W = m.weight.data; o, i = W.shape
        S = W.abs().float() if method == "magnitude" else W.abs().float() * STATS[method][n].sqrt()[None, :]
        if pattern == "2:4":
            idx = S.reshape(o, i // 4, 4).topk(2, dim=-1).indices
            mk = torch.zeros(o, i // 4, 4, dtype=torch.bool, device=W.device).scatter_(-1, idx, True).reshape(o, i)
        else:
            idx = S.topk(i // 2, dim=1).indices
            mk = torch.zeros(o, i, dtype=torch.bool, device=W.device).scatter_(1, idx, True)
        W.mul_(mk.to(W.dtype))

WROWS = {}
for pattern in WANDA_PATTERNS:
    for method in WANDA_METHODS:
        prune(method, pattern); name = f"{'magnitude' if method == 'magnitude' else 'Wanda ' + method} {pattern}"
        WROWS[name] = {ds: run_eval(dense, f"wanda_{method}_{slug(pattern)}_{ds}", items) for ds, items in SETS.items()}
        q = {ds: summarize(WROWS[name][ds].values(), DENSE[ds], boot=False) for ds in SETS}
        print(f"{name:22s} AI2D {q['ai2d']['acc']:5.1f} ({q['ai2d']['d_acc']:+.1f}, flips {q['ai2d']['flip']:.1f}%) | "
              f"POPE {q['pope']['acc']:5.1f} ({q['pope']['d_acc']:+.1f}, flips {q['pope']['flip']:.1f}%)")
restore()
assert all(torch.equal(m.weight.cpu(), ORIG[n]) for n, m in list(TARGETS.items())[:5]), "restore failed"

143 projections, 479M weights


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (458888 > 262144). Running this sequence through the model will result in indexing errors


statistics collected: {'text': 143, 'dec': 143, 'dec_mw': 143}
magnitude 2:4          AI2D  27.6 (-33.2, flips 72.2%) | POPE  49.2 (-31.4, flips 35.8%)
Wanda text 2:4         AI2D  39.8 (-21.0, flips 53.8%) | POPE  81.2 (+0.6, flips 11.0%)
Wanda dec 2:4          AI2D  42.4 (-18.4, flips 49.8%) | POPE  74.6 (-6.0, flips 13.2%)
Wanda dec_mw 2:4       AI2D  40.2 (-20.6, flips 50.2%) | POPE  77.0 (-3.6, flips 12.0%)
magnitude 50%          AI2D  30.2 (-30.6, flips 63.4%) | POPE  62.0 (-18.6, flips 25.8%)
Wanda text 50%         AI2D  49.8 (-11.0, flips 35.4%) | POPE  81.6 (+1.0, flips 6.6%)
Wanda dec 50%          AI2D  50.4 (-10.4, flips 35.2%) | POPE  84.0 (+3.4, flips 5.8%)
Wanda dec_mw 50%       AI2D  51.6 (-9.2, flips 32.2%) | POPE  86.0 (+5.4, flips 7.0%)


### Wanda latency at batch 1
Dense, 2:4-masked weights with ordinary kernels, and 2:4 weights in PyTorch's semi-structured sparse format (real sparse
kernels), timed back to back on the same AI2D items, with dense timed again at the end to show drift.

In [22]:
def block_time(items, warm=3):
    ts = []
    for k, it in enumerate(items):
        torch.cuda.synchronize(); t = time.perf_counter(); ask(dense, it); torch.cuda.synchronize()
        if k >= warm: ts.append(1000 * (time.perf_counter() - t))
    return float(np.median(ts))

its = SETS["ai2d"][:N_LAT + 3]; LATW = {}
restore(); LATW["dense"] = block_time(its)
prune("dec_mw", "2:4"); LATW["2:4 masked, dense kernels"] = block_time(its)
saved = {}
try:
    from torch.sparse import to_sparse_semi_structured
    for n, m in TARGETS.items():
        saved[n] = m.weight; m.weight = torch.nn.Parameter(to_sparse_semi_structured(m.weight.data), requires_grad=False)
    LATW["2:4 sparse kernels"] = block_time(its)
except Exception as ex:
    print("2:4 sparse kernels failed:", repr(ex)[:200])
finally:
    for n, m in TARGETS.items():
        if n in saved: m.weight = saved[n]
restore(); LATW["dense (again)"] = block_time(its)
RESULTS["_wanda_latency_ms"] = LATW; save()
for k, v in LATW.items(): print(f"{k:28s} {v:6.1f} ms ({LATW['dense'] / v:.2f}x)")

/usr/local/lib/python3.13/dist-packages/torch/sparse/semi_structured.py:115: UserWarning: The PyTorch API of SparseSemiStructuredTensor is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.sparse module for further information about the project.
  warnings.warn(


dense                         186.8 ms (1.00x)
2:4 masked, dense kernels     186.2 ms (1.00x)
2:4 sparse kernels            783.3 ms (0.24x)
dense (again)                 188.4 ms (0.99x)


## §13 Every approach in one table
Speedups come from §8 (depth, same session) and §12 (Wanda, AI2D timing applied to both sets). Δ vs FastJEV is a paired bootstrap
on the same questions.

In [23]:
ALL = {"dense": (DENSE, {ds: 1.0 for ds in SETS})}
if FJ_ROWS: ALL["FastJEV S+P 6"] = (FJ_ROWS, SPEED["FastJEV S+P 6"])
for key in PAIRS:
    ALL[f"{key} [static shallow]"] = ({ds: ROWS[("shallow", key, "sweep", ds)] for ds in SETS}, SPEED[f"{key} [static shallow]"])
    ALL[f"{key} [adaptive]"] = (ADAPT[key], SPEED[f"{key} [adaptive]"])
wspeed = LATW["dense"] / LATW["2:4 masked, dense kernels"]
for name, rr in WROWS.items(): ALL[name] = (rr, {ds: wspeed for ds in SETS})

lines = []
for name, (rr, sp) in ALL.items():
    cells = []
    for ds in SETS:
        s = summarize(rr[ds].values(), FJ_ROWS[ds]) if FJ_ROWS and name != "FastJEV S+P 6" else summarize(rr[ds].values())
        vs = f"{s['d_acc']:+.1f} [{s['d_acc_ci'][0]:+.1f}, {s['d_acc_ci'][1]:+.1f}]" if "d_acc" in s else "–"
        cells.append(f"{s['acc']:.1f} | {vs} | {s['nll']:.3f} | {aurc(rr[ds].values()):.1f} | {sp[ds]:.2f}")
    lines.append(f"| {name} | " + " | ".join(cells) + " |")
hdr = ("| Approach | AI2D acc | Δ vs FastJEV | NLL | AURC | speedup | POPE acc | Δ vs FastJEV | NLL | AURC | speedup |\n"
       + "|---" * 11 + "|\n")
display(Markdown(hdr + "\n".join(lines))); open(f"{OUT}/all_approaches.md", "w").write(hdr + "\n".join(lines))
print("saved in", OUT, ":", sorted(os.listdir(OUT)))

| Approach | AI2D acc | Δ vs FastJEV | NLL | AURC | speedup | POPE acc | Δ vs FastJEV | NLL | AURC | speedup |
|---|---|---|---|---|---|---|---|---|---|---|
| dense | 60.8 | -1.0 [-3.4, +1.2] | 0.995 | 20.7 | 1.00 | 80.6 | +1.0 [+0.2, +2.0] | 0.520 | 8.8 | 1.00 |
| FastJEV S+P 6 | 61.8 | – | 0.971 | 19.4 | 1.20 | 79.6 | – | 0.472 | 10.6 | 1.21 |
| dense | p8/b8 [static shallow] | 58.4 | -3.4 [-7.2, +0.6] | 1.047 | 24.4 | 1.34 | 73.6 | -6.0 [-8.6, -3.8] | 0.598 | 13.9 | 1.35 |
| dense | p8/b8 [adaptive] | 59.8 | -2.0 [-4.8, +0.4] | 1.017 | 22.4 | 1.02 | 76.4 | -3.2 [-5.4, -1.2] | 0.577 | 13.0 | 1.35 |
| dense | p6/b6 [static shallow] | 61.4 | -0.4 [-2.6, +1.6] | 0.954 | 19.5 | 1.23 | 77.0 | -2.6 [-4.0, -1.4] | 0.510 | 11.1 | 1.25 |
| dense | p6/b6 [adaptive] | 61.4 | -0.4 [-2.6, +1.8] | 0.956 | 19.5 | 1.23 | 78.2 | -1.4 [-3.0, +0.0] | 0.497 | 10.8 | 1.24 |
| FastJEV 6 | +prefix 6 [static shallow] | 62.4 | +0.6 [-1.0, +2.0] | 0.959 | 19.2 | 1.34 | 78.8 | -0.8 [-1.8, -0.2] | 0.480 | 10.8 | 1.37 |
| FastJEV 6 | +prefix 6 [adaptive] | 62.4 | +0.6 [-1.0, +2.0] | 0.959 | 19.2 | 1.35 | 78.8 | -0.8 [-1.8, -0.2] | 0.480 | 10.8 | 1.36 |
| magnitude 2:4 | 27.6 | -34.2 [-39.8, -28.0] | 1.410 | 69.4 | 1.00 | 49.2 | -30.4 [-34.8, -26.2] | 0.795 | 42.7 | 1.00 |
| Wanda text 2:4 | 39.8 | -22.0 [-27.2, -17.0] | 1.298 | 51.9 | 1.00 | 81.2 | +1.6 [-1.6, +4.6] | 0.479 | 11.5 | 1.00 |
| Wanda dec 2:4 | 42.4 | -19.4 [-24.6, -14.4] | 1.261 | 43.5 | 1.00 | 74.6 | -5.0 [-8.4, -2.0] | 0.530 | 15.0 | 1.00 |
| Wanda dec_mw 2:4 | 40.2 | -21.6 [-26.6, -16.4] | 1.266 | 46.6 | 1.00 | 77.0 | -2.6 [-6.0, +0.4] | 0.498 | 13.1 | 1.00 |
| magnitude 50% | 30.2 | -31.6 [-36.6, -26.4] | 1.385 | 62.5 | 1.00 | 62.0 | -17.6 [-22.0, -13.6] | 0.709 | 29.5 | 1.00 |
| Wanda text 50% | 49.8 | -12.0 [-16.2, -7.8] | 1.155 | 32.6 | 1.00 | 81.6 | +2.0 [-0.4, +4.2] | 0.458 | 9.3 | 1.00 |
| Wanda dec 50% | 50.4 | -11.4 [-15.6, -7.2] | 1.161 | 33.5 | 1.00 | 84.0 | +4.4 [+2.2, +6.4] | 0.425 | 8.7 | 1.00 |
| Wanda dec_mw 50% | 51.6 | -10.2 [-14.0, -6.0] | 1.151 | 32.2 | 1.00 | 86.0 | +6.4 [+3.8, +8.6] | 0.394 | 7.6 | 1.00 |

saved in /content/drive/MyDrive/ese6800_omnijev/results/iter5 : ['adaptive_curves.png', 'adaptive_dense_p6_b6_ai2d.jsonl', 'adaptive_dense_p6_b6_pope.jsonl', 'adaptive_dense_p8_b8_ai2d.jsonl', 'adaptive_dense_p8_b8_pope.jsonl', 'adaptive_fastjev_6_prefix_6_ai2d.jsonl', 'adaptive_fastjev_6_prefix_6_pope.jsonl', 'all_approaches.md', 'deep_dense_sweep_ai2d.jsonl', 'deep_dense_sweep_pope.jsonl', 'deep_dense_train_ai2d.jsonl', 'deep_dense_train_pope.jsonl', 'deep_fj6_sweep_ai2d.jsonl', 'deep_fj6_sweep_pope.jsonl', 'deep_fj6_train_ai2d.jsonl', 'deep_fj6_train_pope.jsonl', 'env.json', 'results.json', 'routing_table.md', 'shallow_dense_p6_b6_sweep_ai2d.jsonl', 'shallow_dense_p6_b6_sweep_pope.jsonl', 'shallow_dense_p6_b6_train_ai2d.jsonl', 'shallow_dense_p6_b6_train_pope.jsonl', 'shallow_dense_p8_b8_sweep_ai2d.jsonl', 'shallow_dense_p8_b8_sweep_pope.jsonl', 'shallow_dense_p8_b8_train_ai2d.jsonl', 'shallow_dense_p8_b8_train_pope.jsonl', 'shallow_fastjev_6_prefix_6_sweep_ai2d.jsonl', 'shallow_fas

## §14 Tien Dee’s experiment — which layers can we skip?

Amit skipped layers 15–20 when scoring the answers and tried also skipping them while reading the image and question. Here we keep those two stages and change **which layers** they skip. If we skip a layer while reading the image and question, its cached state is missing when we score the answers. We therefore also skip that layer during answer scoring.

In a fresh Colab GPU session, run §§0–3 and the **first code cell of §4**, then run §14 from top to bottom. The two lists at the top of the next cell control which layers and group sizes to try. The main comparison stays at six layers to match Amit. Sections 5–13 contain Amit’s earlier experiments.

1. On the **practice questions**, omit one layer at a time. Count how often this changes the full model’s answer. A change counts even if both answers are wrong.
2. Combine the layers with the fewest changes. Check the groups on practice questions and save the top 2, 4, 6 and 8 lists for Sivangi.
3. Freeze the six-layer choice. Compare it with Amit’s six on **different questions**, measuring both accuracy and time per question on the same GPU.

Amit drew both sets from the public test files. We use different question IDs for practice and evaluation on both datasets. POPE also uses different photos in each set.


In [7]:
# Test layers 8 through 22 one at a time. Python stops before 23.
candidate_layers = list(range(8, 23))

# Try these group sizes after ranking the layers. Keep 6 to compare with Amit.
group_sizes = [2, 4, 6, 8]


Run the next cell as written. It checks the question splits and prepares a results folder so an interrupted run can continue.


In [8]:
comparison_size = 6  # fixed for the comparison with Amit’s six layers
timed_questions = 40  # per dataset, after three warmup questions

from copy import copy
from hashlib import sha256
from pathlib import Path
import pandas as pd

original_six = sorted(FJ_MASK["fj6"])
assert original_six == list(range(15, 21))
assert comparison_size in group_sizes and max(group_sizes) <= len(set(candidate_layers))
assert len(candidate_layers) == len(set(candidate_layers))
assert all(L_PROBE <= layer < NL - 1 and layer != FIRST_FULL for layer in candidate_layers)

# The same questions, model revisions, GPU and precision can reuse a previous run.
gpu_name = torch.cuda.get_device_name()
precision = str(next(LAYERS["dense"][0].parameters()).dtype)
run_identity = {"gpu": gpu_name, "precision": precision, "torch": str(torch.__version__),
                "adapter": cfg["artifacts"]["adapter"]["revision"], "base": cfg["artifacts"]["base"]["revision"]}
run_identity["attention"] = "recurrent FLA" if torch.cuda.get_device_capability()[0] < 8 else "chunked FLA"
run_identity["fastjev_revision"] = "638f4eb09eb8f44a7f1e8fcadc666e2ab945c205"
for dataset in ["ai2d", "pope"]:
    practice_ids = [item["id"] for item in TRAIN[dataset]]
    held_out_ids = [item["id"] for item in SETS[dataset]]
    assert len(set(practice_ids)) == len(practice_ids) == 600
    assert len(set(held_out_ids)) == len(held_out_ids) == 500
    assert set(practice_ids).isdisjoint(held_out_ids)
    run_identity[dataset] = {"practice": practice_ids, "held_out": held_out_ids}

practice_photos = {item["request"]["images"][0] for item in TRAIN["pope"]}
held_out_photos = {item["request"]["images"][0] for item in SETS["pope"]}
assert practice_photos.isdisjoint(held_out_photos)

run_key = sha256(json.dumps(run_identity, sort_keys=True).encode()).hexdigest()[:12]
results_folder = Path(OUT) / f"layer_study_{run_key}"
results_folder.mkdir(parents=True, exist_ok=True)
(results_folder / "run_identity.json").write_text(json.dumps(run_identity, indent=2))
print("GPU:", run_identity["gpu"], "| Model precision:", run_identity["precision"])
print("Practice: 600 AI2D + 600 POPE. Held out: 500 AI2D + 500 POPE.")
print("Predictions and tables:", results_folder)


GPU: Tesla T4 | Model precision: torch.float16
Practice: 600 AI2D + 600 POPE. Held out: 500 AI2D + 500 POPE.
Predictions and tables: /content/ese6800_omnijev/results/tien_dee_layer_selection/layer_study_6689f4d4b618


### 1. Rank the layers on practice questions

We test layers 8–22 to stay within the range after Amit’s layer 7 probe and keep the last layer intact.

Record the original full model’s answers once. Then run the **same model** again for each layer, skipping just that layer while reading the input and scoring the candidate answers. Comparing predictions by question ID tells us how many decisions that single change affected.

Each dataset contributes 600 questions, so adding the two flip counts gives equal weight to AI2D and POPE. Fewer flips rank first. We also show practice accuracy for each layer so you can see whether the changed answers helped or hurt. An exact tie goes to the smaller layer index. Accuracy on the held-out questions has no role in this choice.

The `practice_ms` columns show the median time per question for each single-layer run. These runs happen one after another, so use the later interleaved timing for the final speed comparison.

Amit’s `run_eval` saves predictions as it goes. Rerunning a cell with the same setup reuses the completed questions.


In [9]:
def evaluate_setting(split_name, datasets, spec):
    # Include every skipped layer in the file name so a changed mask starts a new run.
    engine_name = spec["eng"]
    masks = [FJ_MASK.get(engine_name, []), spec.get("P", []), spec.get("B", [])]
    name_parts = [engine_name]
    for mask in masks:
        if not mask:
            name_parts.append("none")
            continue
        layer_numbers = []
        for layer in sorted(mask):
            layer_numbers.append(str(layer))
        name_parts.append("-".join(layer_numbers))
    setting_key = "_".join(name_parts)

    rows = {}
    with Route(**spec):
        for dataset, questions in datasets.items():
            tag = f"{results_folder.name}/{split_name}_{dataset}_{setting_key}"
            rows[dataset] = run_eval(ENG[engine_name], tag, questions)
    return rows

dense_spec = {"eng": "dense"}
print("Recording the original full model’s practice answers.", flush=True)
practice_dense = evaluate_setting("practice", TRAIN, dense_spec)
for dataset, rows in practice_dense.items():
    accuracy = summarize(rows.values(), boot=False)["acc"]
    durations = [row["ms"] for row in rows.values()]
    print(f"Full model on {dataset}: {accuracy:.1f}% correct, {np.median(durations):.1f} ms per question.")


Recording the original full model’s practice answers.


Full model on ai2d: 60.0% correct, 193.1 ms per question.
Full model on pope: 84.7% correct, 163.9 ms per question.


In [10]:
layer_results = []
for layer in candidate_layers:
    print(f"Checking layer {layer}...", flush=True)
    # Skip this layer when reading the input (P) and scoring each answer (B).
    spec = {"eng": "dense", "P": [layer], "B": [layer], "mode": "shallow"}
    predictions = evaluate_setting("practice", TRAIN, spec)
    result = {"layer": layer, "type": TYPES[layer]}

    for dataset, rows in predictions.items():
        full_answers = practice_dense[dataset]
        result[f"{dataset}_practice_ms"] = float(np.median([row["ms"] for row in rows.values()]))
        flips = 0
        correct = 0
        for question_id, row in rows.items():
            answer_before = pred(full_answers[question_id]["p"])
            answer_after = pred(row["p"])
            if answer_before != answer_after:
                flips += 1
            if answer_after == row["gold"]:
                correct += 1
        result[f"{dataset}_flips"] = flips
        result[f"{dataset}_accuracy_pct"] = 100 * correct / len(rows)
    result["total_flips"] = result["ai2d_flips"] + result["pope_flips"]
    layer_results.append(result)
    print(f"  {result['ai2d_flips']} AI2D + {result['pope_flips']} POPE changes", flush=True)

ranking = pd.DataFrame(layer_results).sort_values(["total_flips", "layer"]).reset_index(drop=True)
ranking["flip_pct"] = 100 * ranking["total_flips"] / sum(len(items) for items in TRAIN.values())
ranking.to_csv(results_folder / "practice_layer_ranking.csv", index=False)
display(ranking.round(2))


Checking layer 8...


  132 AI2D + 30 POPE changes


Checking layer 9...


  174 AI2D + 20 POPE changes


Checking layer 10...


  161 AI2D + 21 POPE changes


... (21 outputs left out)

  33 AI2D + 0 POPE changes


Checking layer 22...


  13 AI2D + 0 POPE changes


,layer,type,ai2d_practice_ms,ai2d_flips,ai2d_accuracy_pct,pope_practice_ms,pope_flips,pope_accuracy_pct,total_flips,flip_pct
0,22,linear_attention,184.72,13,59.50,151.42,0,84.67,13,1.08
1,18,linear_attention,173.51,14,59.33,168.28,0,84.67,14,1.17
2,17,linear_attention,190.54,12,59.83,158.10,4,84.00,16,1.33
3,20,linear_attention,182.30,16,60.67,153.02,0,84.67,16,1.33
4,19,full_attention,182.20,16,61.17,169.40,1,84.50,17,1.42
5,15,full_attention,170.63,24,60.33,156.86,0,84.67,24,2.00
6,16,linear_attention,181.24,24,58.50,174.41,3,84.17,27,2.25
7,21,linear_attention,175.41,33,60.83,170.67,0,84.67,33,2.75
8,14,linear_attention,185.63,42,59.17,160.23,3,84.17,45,3.75
9,13,linear_attention,194.83,54,59.50,170.76,29,87.83,83,6.92


### 2. Try the highest-ranked layers together

Two layers that were harmless alone may change an answer when both are skipped. We test each group on the same practice questions. The six-layer group is our comparison with Amit because both methods then skip the same number of layers. We keep that choice even if another group looks better.

For grouped and held-out runs, we use FastJEV’s candidate-scoring implementation so our masks and Amit’s mask run through the same code. FastJEV normally reads the skipped layers from a published configuration. For each new group, we give its existing engine another mask. The groups share the same model weights. For the first stage we reuse Amit’s `Route`.


In [11]:
def use_candidate_mask(name, layers):
    # Same FastJEV engine and weights as Amit. Only the list used in candidate scoring changes.
    ENG[name] = copy(ENG["fj6"])
    ENG[name].layers = layers
    LAYERS[name] = LAYERS["fj6"]
    FJ_MASK[name] = layers

# Freeze and save the choices before opening any held-out predictions.
ranked_layers = ranking["layer"].tolist()
chosen_layers = {size: sorted(ranked_layers[:size]) for size in group_sizes}
(results_folder / "chosen_layers.json").write_text(json.dumps(chosen_layers, indent=2))

ranked_engines = {}
practice_groups = []
for size, layers in chosen_layers.items():
    name = f"ranked{size}"
    ranked_engines[size] = name
    use_candidate_mask(name, layers)

    spec = {"eng": name, "P": layers, "mode": "shallow"}
    print(f"Practice check with {size} layers: {layers}", flush=True)
    predictions = evaluate_setting("practice", TRAIN, spec)
    for dataset, rows in predictions.items():
        scores = summarize(rows.values(), ref=practice_dense[dataset], boot=False)
        practice_groups.append({"count": size, "layers": layers, "dataset": dataset,
                                "changed_pct": scores["flip"], "accuracy_pct": scores["acc"]})

practice_table = pd.DataFrame(practice_groups)
practice_table.to_csv(results_folder / "practice_layer_groups.csv", index=False)
display(practice_table.round(2))


Practice check with 2 layers: [18, 22]


Practice check with 4 layers: [17, 18, 20, 22]

Practice check with 6 layers: [15, 17, 18, 19, 20, 22]


Practice check with 8 layers: [15, 16, 17, 18, 19, 20, 21, 22]


,count,layers,dataset,changed_pct,accuracy_pct
0,2,"[18, 22]",ai2d,2.67,60.17
1,2,"[18, 22]",pope,0.17,84.50
2,4,"[17, 18, 20, 22]",ai2d,4.33,61.17
3,4,"[17, 18, 20, 22]",pope,2.33,82.67
4,6,"[15, 17, 18, 19, 20, 22]",ai2d,5.67,60.50
5,6,"[15, 17, 18, 19, 20, 22]",pope,1.50,83.50
6,8,"[15, 16, 17, 18, 19, 20, 21, 22]",ai2d,6.67,61.00
7,8,"[15, 16, 17, 18, 19, 20, 21, 22]",pope,7.00,78.00


### 3. Compare the two six-layer choices on held-out questions

**Branches only** skips layers while scoring the candidate answers. **Prefix and branches** skips those same layers when reading the image and question as well. The two masks use the same FastJEV candidate-scoring code. This gives us two comparisons. Within either stage setting we change only the mask. Within either mask we can see what adding the prefix skip does.

Here the dense baseline uses **Amit’s original engine** so the accuracy and speed comparisons match his benchmark. Both six-layer settings still use the same FastJEV candidate scoring.

Accuracy is the percentage of questions answered correctly. The change from dense is measured in percentage points. Each row uses all 500 held-out questions for that dataset.


In [12]:
ranked_name = ranked_engines[comparison_size]
ranked_six = chosen_layers[comparison_size]
settings = {
    "Full model": dense_spec,
    "Amit 6, branches only": {"eng": "fj6"},
    "Amit 6, prefix and branches": {"eng": "fj6", "P": original_six, "mode": "shallow"},
    "Ranked 6, branches only": {"eng": ranked_name},
    "Ranked 6, prefix and branches": {"eng": ranked_name, "P": ranked_six, "mode": "shallow"},
}
print("Amit’s layers:", original_six, "| Ranked layers:", ranked_six)

held_out_dense = evaluate_setting("held_out", SETS, dense_spec)
accuracy_results = []
for label, spec in settings.items():
    print(label, flush=True)
    predictions = held_out_dense if label == "Full model" else evaluate_setting("held_out", SETS, spec)
    for dataset, rows in predictions.items():
        scores = summarize(rows.values(), ref=held_out_dense[dataset], boot=False)
        accuracy_results.append({"setting": label, "dataset": dataset, "questions": len(rows),
                                 "accuracy_pct": scores["acc"], "change_vs_dense_pp": scores["d_acc"],
                                 "changed_answers_pct": scores["flip"]})
accuracy_table = pd.DataFrame(accuracy_results)
accuracy_table.to_csv(results_folder / "held_out_accuracy.csv", index=False)
display(accuracy_table.round(2))


Amit’s layers: [15, 16, 17, 18, 19, 20] | Ranked layers: [15, 17, 18, 19, 20, 22]


Full model


Amit 6, branches only


Amit 6, prefix and branches


Ranked 6, branches only


Ranked 6, prefix and branches


,setting,dataset,questions,accuracy_pct,change_vs_dense_pp,changed_answers_pct
0,Full model,ai2d,500,61.6,0.0,0.0
1,Full model,pope,500,80.4,0.0,0.0
2,"Amit 6, branches only",ai2d,500,62.4,0.8,8.8
3,"Amit 6, branches only",pope,500,79.2,-1.2,1.2
4,"Amit 6, prefix and branches",ai2d,500,62.2,0.6,7.8
5,"Amit 6, prefix and branches",pope,500,78.6,-1.8,1.8
6,"Ranked 6, branches only",ai2d,500,63.0,1.4,8.2
7,"Ranked 6, branches only",pope,500,79.0,-1.4,1.4
8,"Ranked 6, prefix and branches",ai2d,500,63.2,1.6,8.0
9,"Ranked 6, prefix and branches",pope,500,78.0,-2.4,2.4


### 4. Time the five settings on the same Colab GPU

We use Amit’s §8 procedure on each dataset. Every setting answers the same question before we advance. The first setting rotates with each question. We discard the first three questions as warmup, then take the median of 40 individual request times. A speedup of 1.20 means the full model took 1.20 times as long as that setting. Compare with Amit’s six using the rows from this run.


In [13]:
# Same synchronized single-request timer as Amit’s §8.
def time_one(spec, it):
    with Route(**spec):
        torch.cuda.synchronize()
        start = time.perf_counter()
        ask(ENG[spec["eng"]], it)
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start)

timings = []
labels = list(settings)
for dataset, items in SETS.items():
    print(f"Timing all settings on {dataset}.", flush=True)
    for index, item in enumerate(items[:3 + timed_questions]):
        start = index % len(labels)
        for label in labels[start:] + labels[:start]:
            elapsed_ms = time_one(settings[label], item)
            if index >= 3:
                timings.append({"setting": label, "dataset": dataset, "id": item["id"], "ms": elapsed_ms})

raw_times = pd.DataFrame(timings)
raw_times.to_csv(results_folder / "held_out_timing_requests.csv", index=False)
median_ms = raw_times.groupby(["dataset", "setting"])["ms"].median()
final_table = accuracy_table.copy()
final_table["median_ms"] = [median_ms[dataset, label] for dataset, label in zip(final_table.dataset, final_table.setting)]
final_table["speedup_vs_dense"] = [median_ms[dataset, "Full model"] / ms
                                   for dataset, ms in zip(final_table.dataset, final_table.median_ms)]
final_table.to_csv(results_folder / "held_out_results.csv", index=False)
display(final_table.round(3))


Timing all settings on ai2d.


Timing all settings on pope.


,setting,dataset,questions,accuracy_pct,change_vs_dense_pp,changed_answers_pct,median_ms,speedup_vs_dense
0,Full model,ai2d,500,61.6,0.0,0.0,175.417,1.000
1,Full model,pope,500,80.4,0.0,0.0,167.294,1.000
2,"Amit 6, branches only",ai2d,500,62.4,0.8,8.8,168.353,1.042
3,"Amit 6, branches only",pope,500,79.2,-1.2,1.2,153.869,1.087
4,"Amit 6, prefix and branches",ai2d,500,62.2,0.6,7.8,152.485,1.150
5,"Amit 6, prefix and branches",pope,500,78.6,-1.8,1.8,142.406,1.175
6,"Ranked 6, branches only",ai2d,500,63.0,1.4,8.2,165.664,1.059
7,"Ranked 6, branches only",pope,500,79.0,-1.4,1.4,153.364,1.091
8,"Ranked 6, prefix and branches",ai2d,500,63.2,1.6,8.0,149.322,1.175
9,"Ranked 6, prefix and branches",pope,500,78.0,-2.4,2.4,138.944,1.204


### What happened on the T4

The six layers selected on practice questions were **15, 17, 18, 19, 20, 22**. Amit used **15, 16, 17, 18, 19, 20**. We fixed both lists before measuring held-out accuracy.

The ranking replaced layer 16 with layer 22. On its own, skipping layer 16 changed 27 of 1,200 practice answers. Skipping layer 22 changed 13. Both are linear-attention layers, so the two six-layer lists still skip the same full-attention layers, 15 and 19.

| Held-out task | Amit’s six, accuracy | Ranked six, accuracy | Amit’s six, median time | Ranked six, median time |
| --- | ---: | ---: | ---: | ---: |
| AI2D | 62.2% | 63.2% | 152.5 ms | 149.3 ms |
| POPE | 78.6% | 78.0% | 142.4 ms | 138.9 ms |

On AI2D, the new list answered five more questions correctly. On POPE, it answered three fewer correctly. Median time fell by about 3 ms on both datasets in this run.

Both columns skip the six layers in both stages. Each accuracy uses 500 questions. Each time is the median of 40 requests after warmup. The preceding table also shows the full model and the two settings that skip layers only while scoring answers.

**Compute.** About 94 minutes elapsed from the first practice prediction through the final timing run on one Google Colab Tesla T4, including one restart to repair a cached image. All 29,000 saved predictions and 400 timed requests were checked against the reported tables. The T4 used FP16 and the recurrent implementation of the gated delta rule in the linear-attention layers. Compare timings within this run. Amit’s archived results above used an L4.
